In [ ]:
# ========== 第 1 周 Day2 练习：用本地 Ollama 做网页摘要 ==========
# 练习目标：抓网页正文 → 拼 messages → 调本地模型 → 用 Markdown 展示摘要
# 和本课关系：Chat Completions、system/user 角色、本地 OpenAI 兼容端点（本格假定已有 ollama 客户端）

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key（本练习主路径走本地 Ollama）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从同目录 scraper 模块导入 fetch_website_contents：抓取并清洗网页正文（需本目录有 scraper.py）
from scraper import fetch_website_contents
# 从 IPython.display 导入 Markdown、display：在笔记本里漂亮地渲染模型返回的 Markdown
from IPython.display import Markdown, display

# system_prompt：给模型的角色与规则（发给模型的指令字符串保持英文，改译会改变行为）
system_prompt = """
You are a assistant that analyzes the contents of a website,
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.

"""

# user_prompt_prefix：用户消息前缀；真正网页正文会拼在后面
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# messages_for：把「系统设定 + 用户内容」打成 Chat Completions 需要的 messages 列表
def messages_for(website):
    return [
        # system：规定摘要风格与输出格式
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 网页正文
        {"role": "user", "content": user_prompt_prefix + website}
    ]

# summarize：抓取 URL → 调本地模型 → 返回摘要文本
def summarize(url):
    # 抓取目标网站正文（字符串）；具体清洗逻辑在 scraper.fetch_website_contents 里
    website = fetch_website_contents(url)
    # 调用 Chat Completions：model 名必须与本机 ollama pull 的一致；messages 用上面组装的列表
    # 注意：此处使用名为 ollama 的客户端对象（需在前面单元格/环境中已创建，逻辑保持原样不改）
    response = ollama.chat.completions.create(model="llama3.2", messages=messages_for(website))
    # 从 choices[0].message.content 取出助手回复正文
    return response.choices[0].message.content

# display_summary：封装「摘要 + 展示」两步，方便一键调用
def display_summary(url):
    # 先拿到摘要字符串
    summary = summarize(url)
    # 用 Markdown 渲染后显示在输出区
    display(Markdown(summary))

# 对 CNN 首页做一次端到端演示（URL 字符串保持原样）
display_summary("https://www.cnn.com")

